# 01 - Inventário dos Dados Brutos e Auditoria do Dicionário
**Projeto:** Vitaliza (Módulo 2 MBA Inteli)  
**Autor:** Antigravity AI  
**Data:** 22/10/2026  
**Ambiente:** `dev`  

---
## Objetivo
Executar a auditoria e inventário completo dos 12 arquivos tabulares e dos pacotes de mapas de calor contidos em `dados/brutos/`, comparando metadados reais (linhas, colunas, tipos de dados e valores vazios) com a especificação formal em `docs/Vitaliza - Dicionário de dados.pdf`.


In [1]:
import os
import zipfile
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

RAW_DIR = "../dados/brutos"
print(f"Diretório de dados brutos: {os.path.abspath(RAW_DIR)}")


Diretório de dados brutos: C:\Projetos\Vitaliza\dados\brutos


## 1. Dicionário de Especificações Esperadas
Definição dos metadados extraídos formalmente do `Vitaliza: dicionário de dados`.


In [2]:
expected_dict = {
    "usuarios.csv": {
        "linhas": 64948,
        "colunas": ["user_id", "email", "data_cadastro", "data_nascimento", "sexo", "cidade", "uf", "cep_prefixo", "canal_aquisicao", "sistema", "versao_app", "relogio_conectado"]
    },
    "assinaturas.csv": {
        "linhas": 32643,
        "colunas": ["assinatura_id", "user_id", "plano_atual", "canal_cobranca", "inicio", "status_atual", "renovacao_automatica", "data_fim", "motivo_fim"]
    },
    "amizades.csv": {
        "linhas": 113480,
        "colunas": ["seguidor_id", "seguido_id", "desde"]
    },
    "campanha_ago26.csv": {
        "linhas": 14120,
        "colunas": ["user_id", "grupo", "enviado_em", "canal", "titulo_mensagem", "oferta", "abriu_push", "abriu_email", "clicou"]
    },
    "eventos_assinatura.csv": {
        "linhas": 275565,
        "colunas": ["user_id", "data_hora", "evento", "plano", "canal_cobranca", "detalhe"]
    },
    "pagamentos.csv": {
        "linhas": 165548,
        "colunas": ["pagamento_id", "user_id", "data_hora", "valor", "metodo", "status", "tentativa"]
    },
    "uso_mensal.csv": {
        "linhas": 591662,
        "colunas": ["user_id", "mes", "atividades", "atividades_ar_livre", "atividades_video", "minutos", "dias_ativos", "sessoes_app"]
    },
    "atividades.parquet": {
        "linhas": 1254475,
        "colunas": ["atividade_id", "user_id", "inicio", "tipo", "duracao_min", "distancia_km", "ritmo_min_km", "fc_media", "origem", "gps_ok", "treino_com"]
    },
    "onboarding.csv": {
        "linhas": 63929,
        "colunas": ["onboarding_id", "user_id", "respondido_em", "objetivo_texto", "como_conheceu", "nivel", "dias_por_semana"]
    },
    "pesquisa_saida.csv": {
        "linhas": 643,
        "colunas": ["resposta_id", "user_id", "respondido_em", "motivo_principal", "comentario"]
    },
    "feed_posts.csv": {
        "linhas": 119309,
        "colunas": ["post_id", "user_id", "data_hora", "texto", "curtidas"]
    },
    "tickets_suporte.csv": {
        "linhas": 4422,
        "colunas": ["ticket_id", "user_id", "aberto_em", "canal", "texto_cliente", "categoria_atendente", "status", "resolvido_em"]
    }
}


## 2. Execução do Inventário e Perfilamento dos Arquivos Tabulares
Cálculo de contagem exata de linhas, conformidade de colunas, dtypes e quantificação de valores ausentes/vazios.


In [3]:
summary_rows = []
detailed_profiles = {}

for filename, spec in expected_dict.items():
    filepath = os.path.join(RAW_DIR, filename)
    if filename.endswith(".csv"):
        df = pd.read_csv(filepath, low_memory=False)
    else:
        df = pd.read_parquet(filepath)
        
    n_rows = len(df)
    n_cols = len(df.columns)
    
    colunas_reais = list(df.columns)
    missing_cols = [c for c in spec["colunas"] if c not in colunas_reais]
    extra_cols = [c for c in colunas_reais if c not in spec["colunas"]]
    
    summary_rows.append({
        "Arquivo": filename,
        "Linhas Reais": n_rows,
        "Linhas Esperadas": spec["linhas"],
        "Diff Linhas": n_rows - spec["linhas"],
        "Cols Reais": n_cols,
        "Cols Esperadas": len(spec["colunas"]),
        "Cols Faltando": len(missing_cols),
        "Cols Extras": len(extra_cols)
    })
    
    # Detalhamento de colunas
    col_stats = []
    for col in df.columns:
        s = df[col]
        n_null = int(s.isna().sum())
        if s.dtype == 'object' or str(s.dtype) == 'string':
            s_str = s.fillna('').astype(str).str.strip()
            n_blank = int((s_str == '').sum()) - n_null
            if n_blank < 0: n_blank = 0
        else:
            n_blank = 0
        total_missing = n_null + n_blank
        col_stats.append({
            "Coluna": col,
            "Tipo": str(s.dtype),
            "NaNs": n_null,
            "Strings Vazias": n_blank,
            "Total Ausente": total_missing,
            "% Ausente": round((total_missing / n_rows) * 100, 2)
        })
    detailed_profiles[filename] = pd.DataFrame(col_stats)

df_summary = pd.DataFrame(summary_rows)
display(df_summary)


,Arquivo,Linhas Reais,Linhas Esperadas,Diff Linhas,Cols Reais,Cols Esperadas,Cols Faltando,Cols Extras
0,usuarios.csv,64948,64948,0,12,12,0,0
1,assinaturas.csv,32643,32643,0,9,9,0,0
2,amizades.csv,113480,113480,0,3,3,0,0
3,campanha_ago26.csv,14120,14120,0,9,9,0,0
4,eventos_assinatura.csv,275565,275565,0,6,6,0,0
5,pagamentos.csv,165548,165548,0,7,7,0,0
6,uso_mensal.csv,591662,591662,0,8,8,0,0
7,atividades.parquet,1254475,1254475,0,11,11,0,0
8,onboarding.csv,63929,63929,0,7,7,0,0
9,pesquisa_saida.csv,643,643,0,5,5,0,0


## 3. Inventário dos Mapas de Calor (`heatmaps`)
Inspeção dos pacotes `.zip` contendo os heatmaps de 90 dias em 31/07/2026 e 30/09/2026.


In [4]:
heatmaps_summary = []
hdir = os.path.join(RAW_DIR, "heatmaps")

for zfile in sorted(os.listdir(hdir)):
    if zfile.endswith(".zip"):
        zpath = os.path.join(hdir, zfile)
        with zipfile.ZipFile(zpath, 'r') as z:
            pngs = [n for n in z.namelist() if n.lower().endswith(".png")]
            data_str = zfile.replace("heatmaps_", "").replace(".zip", "")
            esp = 12426 if "2026-07-31" in zfile else 13047
            heatmaps_summary.append({
                "Data Corte": data_str,
                "Arquivo": zfile,
                "PNGs Reais": len(pngs),
                "PNGs Esperados": esp,
                "Diferença": len(pngs) - esp,
                "Exemplo": pngs[0] if pngs else None
            })

df_heatmaps = pd.DataFrame(heatmaps_summary)
display(df_heatmaps)


,Data Corte,Arquivo,PNGs Reais,PNGs Esperados,Diferença,Exemplo
0,2026-07-31,heatmaps_2026-07-31.zip,12431,12426,5,2026-07-31/u_0aa0410a.png
1,2026-09-30,heatmaps_2026-09-30.zip,13047,13047,0,2026-09-30/u_0aa0410a.png


In [5]:
# 4.1 Inconsistência em pagamentos.csv: formato de valores (regex=False para buscar literal 'R$')
df_pag = pd.read_csv(os.path.join(RAW_DIR, "pagamentos.csv"), low_memory=False)
mask_rs = df_pag['valor'].astype(str).str.contains('R$', regex=False)
print(f"Total pagamentos com 'R$': {mask_rs.sum():,} de {len(df_pag):,}")
print("Distribuição por método:")
print(df_pag[mask_rs]['metodo'].value_counts())


Total pagamentos com 'R$': 19,750 de 165,548
Distribuição por método:
metodo
cartao            13044
pix_automatico     6706
Name: count, dtype: int64


In [6]:
# 4.2 Inconsistência em atividades.parquet: treinos solo vs companhia
df_ativ = pd.read_parquet(os.path.join(RAW_DIR, "atividades.parquet"))
n_solo = (df_ativ['treino_com'] == '').sum()
print(f"Treinos solo (string vazia ''): {n_solo:,} ({n_solo/len(df_ativ)*100:.2f}%)")
print(f"Treinos com parceiro: {(df_ativ['treino_com'] != '').sum():,}")


Treinos solo (string vazia ''): 1,240,338 (98.87%)
Treinos com parceiro: 14,137


---
## 5. População Oficial no Corte (`30/09/2026`), Contas Internas e Regra "Juntar sem Multiplicar"
**Racional (Aulas 1 e 2 de Tecnologia):**
- A base `usuarios.csv` possui **64.948 contas cadastradas**, incluindo **60 contas internas (`@vitaliza.com.br` / plano `cortesia`)** que devem ser expurgadas.
- Em `assinaturas.csv`, existem **14.813 assinaturas ativas brutas**; expurgando as 60 contas internas, chegamos exatamente aos **14.753 assinantes pagantes ativos** no corte de `30/09/2026` (sendo **12.503** com `renovacao_automatica == 'sim'` — população-alvo da previsão para `29/11/2026` — e **2.250** com `renovacao_automatica == 'nao'`).
- **Granularidade (*Tidy Data* — Wickham, 2014):** Ao cruzar `usuarios.csv` (`64.948` linhas) com logs de eventos (`atividades.parquet` com `1.254.475` linhas), **nunca fazemos `JOIN` direto 1:N** (o que multiplicaria a tabela para `1.282.872` linhas). Primeiro agregamos por `user_id` e conferimos se o total pós-join permanece em **64.948 contas** (`36.551` com treino e `28.397` sem treino).


In [7]:
df_usr = pd.read_csv(os.path.join(RAW_DIR, 'usuarios.csv'))
df_ass = pd.read_csv(os.path.join(RAW_DIR, 'assinaturas.csv'))

contas_internas = set(df_usr[df_usr['email'].str.endswith('@vitaliza.com.br', na=False)]['user_id'])
ativas_brutas = df_ass[df_ass['status_atual'] == 'ativa']
ativas_pagantes = ativas_brutas[(~ativas_brutas['user_id'].isin(contas_internas)) & (ativas_brutas['plano_atual'] != 'cortesia')]
ativas_renov_sim = ativas_pagantes[ativas_pagantes['renovacao_automatica'] == 'sim']

# Verificação de granularidade: Juntar sem multiplicar (usuarios x atividades)
contagem_treinos_por_user = df_ativ.groupby('user_id').size().rename('qtd_treinos')
df_usr_join = df_usr[['user_id']].merge(contagem_treinos_por_user, on='user_id', how='left')
df_usr_join['qtd_treinos'] = df_usr_join['qtd_treinos'].fillna(0).astype(int)

resumo_pop = pd.DataFrame([
    {'Métrica de Controle': 'Contas cadastradas (usuarios.csv)', 'Valor Encontrado': len(df_usr), 'Referência Oficial (Aula 2)': 64948},
    {'Métrica de Controle': 'Contas internas (@vitaliza.com.br)', 'Valor Encontrado': len(contas_internas), 'Referência Oficial (Aula 2)': 60},
    {'Métrica de Controle': 'Assinaturas ativas brutas', 'Valor Encontrado': len(ativas_brutas), 'Referência Oficial (Aula 2)': 14813},
    {'Métrica de Controle': 'Assinantes pagantes ativos (30/09/2026)', 'Valor Encontrado': len(ativas_pagantes), 'Referência Oficial (Aula 2)': 14753},
    {'Métrica de Controle': 'Pagantes ativos c/ renovação ligada (População Alvo)', 'Valor Encontrado': len(ativas_renov_sim), 'Referência Oficial (Aula 2)': 12503},
    {'Métrica de Controle': 'Linhas pós-JOIN agregado (usuarios + atividades)', 'Valor Encontrado': len(df_usr_join), 'Referência Oficial (Aula 2)': 64948},
    {'Métrica de Controle': 'Contas com >= 1 treino registrado', 'Valor Encontrado': int((df_usr_join['qtd_treinos'] > 0).sum()), 'Referência Oficial (Aula 2)': 36551},
    {'Métrica de Controle': 'Contas com 0 treinos registrados', 'Valor Encontrado': int((df_usr_join['qtd_treinos'] == 0).sum()), 'Referência Oficial (Aula 2)': 28397},
])
display(resumo_pop)


,Métrica de Controle,Valor Encontrado,Referência Oficial (Aula 2)
0,Contas cadastradas (usuarios.csv),64948,64948
1,Contas internas (@vitaliza.com.br),60,60
2,Assinaturas ativas brutas,14813,14813
3,Assinantes pagantes ativos (30/09/2026),14753,14753
4,Pagantes ativos c/ renovação ligada (População...,12503,12503
5,Linhas pós-JOIN agregado (usuarios + atividades),64948,64948
6,Contas com >= 1 treino registrado,36551,36551
7,Contas com 0 treinos registrados,28397,28397


---
## 6. Pipeline das 5 Dimensões da Qualidade de Dados (Wang & Strong, 1996)
**Racional (Aula 2 de Tecnologia — *"Limpar é decidir: Corrigir, Remover ou Marcar"*):**
1. **Completude (`onboarding.csv`):** Separar vazios declarados como opcionais (`objetivo_texto`: `5.467` vazios / `8,6%`) de vazios não declarados como opcionais (`nivel`: `4.513` / `7,1%`; `dias_por_semana`: `6.368` / `10,0%`; `como_conheceu`: `4.586` / `7,2%`). Decisão: **Marcar** com flags indicadoras (`flag_onb_*_preenchido`), sem descartar linhas.
2. **Validade (`atividades.parquet`):** Dos `1.254.475` treinos, `574.835` não possuem `distancia_km` porque são treinos *indoor* legítimos (sem GPS). Nos `679.640` treinos com GPS, há **951 treinos acima de 200 km** (o maior com `10.012,75 km` do `apple_health`) e **3.649 treinos com duração zero**. Remover os `555` erros `> 200 km` nas corridas faz a correlação $r$ de Pearson entre duração e distância saltar de **`0,12` para `0,89`**.
3. **Consistência (`atividades.parquet`):** Existem **11 rótulos para 7 modalidades** (`corrida`/`run`, `caminhada`/`walk`, `forca`/`strength`, `pedal`/`ride`, além de `funcional`, `hiit` e `yoga`). Os **376.930 treinos em inglês (`30,0%`)** vêm todos da origem `app`. Sem mapear `run` $\rightarrow$ `corrida`, perderíamos **162.108 corridas**.
4. **Unicidade (`atividades.parquet`):** Existem **9.383 pares de treinos duplicados** (mesma conta, modalidade canônica, duração, distância e início em até 5 min entre `app` e relógio): **4.184 com `apple_health`** e **5.199 com `garmin`**. *Armadilha crítica:* O `garmin` grava em `UTC` (+3h em relação a Brasília); é obrigatório subtrair 3 horas do `garmin` antes de deduplicar.
5. **Temporalidade (`uso_mensal.csv`):** Em **2025**, mês sem treino era **linha ausente** (`2025-12` tem `13.624` linhas e `0` zeros, média `264,6 min`); a partir de **jan/2026**, mês sem treino virou **linha explícita com zero** (`2026-01` tem `41.130` linhas, sendo `25.697` com zero, derrubando a média bruta para `117,9 min`, enquanto a média de quem treinou subiu para `314,2 min`). Em `2026-09` (`56.565` contas), `72,3%` têm zero minutos (mediana `0`, média `82,6 min`, $s = 221,3\text{ min}$, $CV = 268\%$), e a correlação entre `sessoes_app` e `minutos` (`r = 0,88` nos ativos / `0,92` no geral) cai para **`r = 0,02`** quando controlada pelo número exato de `atividades`.


In [8]:
# Dimensões 2, 3 e 4 em atividades.parquet
MAPA_MODALIDADES = {
    'corrida': 'corrida', 'run': 'corrida',
    'caminhada': 'caminhada', 'walk': 'caminhada',
    'forca': 'forca', 'strength': 'forca',
    'pedal': 'pedal', 'ride': 'pedal',
    'yoga': 'yoga', 'hiit': 'hiit', 'funcional': 'funcional'
}
df_ativ['tipo_canonico'] = df_ativ['tipo'].map(MAPA_MODALIDADES)

# Efeito dos 555 outliers (>200 km) no r de Pearson das corridas
corridas_gps = df_ativ[(df_ativ['tipo_canonico'] == 'corrida') & (df_ativ['distancia_km'].notna())]
r_corr_bruto = corridas_gps['duracao_min'].corr(corridas_gps['distancia_km'])
corridas_limpas = corridas_gps[(corridas_gps['distancia_km'] <= 200) & (corridas_gps['duracao_min'] > 0)]
r_corr_limpo = corridas_limpas['duracao_min'].corr(corridas_limpas['distancia_km'])

# Ajuste de fuso UTC (-3h) para Garmin e detecção dos 9.383 pares duplicados
df_ativ['inicio_dt_br'] = pd.to_datetime(df_ativ['inicio'])
mask_garmin = df_ativ['origem'] == 'garmin'
df_ativ.loc[mask_garmin, 'inicio_dt_br'] = df_ativ.loc[mask_garmin, 'inicio_dt_br'] - pd.Timedelta(hours=3)

df_sorted = df_ativ.sort_values(['user_id', 'tipo_canonico', 'duracao_min', 'inicio_dt_br']).copy()
diff_min = (df_sorted['inicio_dt_br'] - df_sorted['inicio_dt_br'].shift(1)).dt.total_seconds().abs() / 60.0
is_dup = (
    (df_sorted['user_id'] == df_sorted['user_id'].shift(1))
    & (df_sorted['tipo_canonico'] == df_sorted['tipo_canonico'].shift(1))
    & (df_sorted['duracao_min'] == df_sorted['duracao_min'].shift(1))
    & (np.isclose(df_sorted['distancia_km'].fillna(-1), df_sorted['distancia_km'].fillna(-1).shift(1)))
    & (diff_min <= 5.0)
    & (df_sorted['origem'] != df_sorted['origem'].shift(1))
)
relogio_par = pd.Series(np.where(df_sorted['origem'] == 'app', df_sorted['origem'].shift(1), df_sorted['origem']))[is_dup.values]
dups_relogio = relogio_par.value_counts().to_dict()

# Dimensão 5 em uso_mensal.csv
df_uso = pd.read_csv(os.path.join(RAW_DIR, 'uso_mensal.csv'))
dez25 = df_uso[df_uso['mes'] == '2025-12']
jan26 = df_uso[df_uso['mes'] == '2026-01']
set26 = df_uso[df_uso['mes'] == '2026-09']
set26_at = set26[set26['minutos'] > 0]

corrs_k, pesos_k = [], []
for k, grp in set26_at.groupby('atividades'):
    if len(grp) >= 30 and grp['sessoes_app'].std() > 0 and grp['minutos'].std() > 0:
        corrs_k.append(grp['sessoes_app'].corr(grp['minutos']))
        pesos_k.append(len(grp))
r_parcial_set26 = float(np.average(corrs_k, weights=pesos_k))

tabela_5d = pd.DataFrame([
    {'Dimensão': '1. Completude', 'Achado Auditado': 'onboarding.csv: vazios em objetivo_texto (opcional) vs dias_por_semana (não declarado)', 'Valor Encontrado': f"{detailed_profiles['onboarding.csv'].loc[3, 'Total Ausente']:,} (8.6%) vs {detailed_profiles['onboarding.csv'].loc[6, 'Total Ausente']:,} (10.0%)", 'Ref. Aula 2': '5.467 (8,6%) vs 6.368 (10,0%)'},
    {'Dimensão': '2. Validade', 'Achado Auditado': 'atividades.parquet: sem GPS (indoor) | > 200 km | duração <= 0', 'Valor Encontrado': f"{int(df_ativ['distancia_km'].isna().sum()):,} | {int((df_ativ['distancia_km']>200).sum()):,} | {int((df_ativ['duracao_min']<=0).sum()):,}", 'Ref. Aula 2': '574.835 | 951 | 3.649'},
    {'Dimensão': '2. Validade (r)', 'Achado Auditado': 'Corridas: r(duracao, distancia) antes vs depois de remover 555 treinos > 200 km', 'Valor Encontrado': f"{r_corr_bruto:.2f} -> {r_corr_limpo:.2f}", 'Ref. Aula 2': '0,12 -> 0,88+'},
    {'Dimensão': '3. Consistência', 'Achado Auditado': 'atividades.parquet: treinos em inglês (origem app) | corridas "run" resgatadas', 'Valor Encontrado': f"{int(df_ativ['tipo'].isin(['run','walk','strength','ride']).sum()):,} (30.0%) | {int((df_ativ['tipo']=='run').sum()):,}", 'Ref. Aula 2': '376.930 (30%) | 162.108'},
    {'Dimensão': '4. Unicidade', 'Achado Auditado': 'Pares duplicados app + relógio (após converter Garmin UTC -> UTC-3)', 'Valor Encontrado': f"{int(is_dup.sum()):,} (Garmin: {dups_relogio.get('garmin',0):,}, Apple: {dups_relogio.get('apple_health',0):,})", 'Ref. Aula 2': '9.383 (Garmin: 5.199, Apple: 4.184)'},
    {'Dimensão': '5. Temporalidade', 'Achado Auditado': 'uso_mensal.csv: média bruta Dez/25 -> Jan/26 vs média ativos (>0 min) Jan/26', 'Valor Encontrado': f"{dez25['minutos'].mean():.0f} min -> {jan26['minutos'].mean():.0f} min (ativos: {jan26[jan26['minutos']>0]['minutos'].mean():.0f} min)", 'Ref. Aula 2': '265 min -> 118 min (ativos: 314 min)'},
    {'Dimensão': '5. Controle (Set/26)', 'Achado Auditado': 'r(sessoes_app, minutos) geral vs ativos vs controlado por atividades exatas', 'Valor Encontrado': f"{set26['sessoes_app'].corr(set26['minutos']):.2f} | {set26_at['sessoes_app'].corr(set26_at['minutos']):.2f} | {r_parcial_set26:.2f}", 'Ref. Aula 2': '0,92 | 0,88 | 0,02'},
])
display(tabela_5d)


,Dimensão,Achado Auditado,Valor Encontrado,Ref. Aula 2
0,1. Completude,onboarding.csv: vazios em objetivo_texto (opci...,"5,467 (8.6%) vs 6,368 (10.0%)","5.467 (8,6%) vs 6.368 (10,0%)"
1,2. Validade,atividades.parquet: sem GPS (indoor) | > 200 k...,"574,835 | 951 | 3,649",574.835 | 951 | 3.649
2,2. Validade (r),"Corridas: r(duracao, distancia) antes vs depoi...",0.12 -> 0.89,"0,12 -> 0,88+"
3,3. Consistência,atividades.parquet: treinos em inglês (origem ...,"376,930 (30.0%) | 162,108",376.930 (30%) | 162.108
4,4. Unicidade,Pares duplicados app + relógio (após converter...,"9,383 (Garmin: 5,199, Apple: 4,184)","9.383 (Garmin: 5.199, Apple: 4.184)"
5,5. Temporalidade,uso_mensal.csv: média bruta Dez/25 -> Jan/26 v...,265 min -> 118 min (ativos: 314 min),265 min -> 118 min (ativos: 314 min)
6,5. Controle (Set/26),"r(sessoes_app, minutos) geral vs ativos vs con...",0.92 | 0.88 | 0.02,"0,92 | 0,88 | 0,02"


---
## 7. Prevenção de Vazamento Temporal (*Data Leakage*) e Reconciliação Contábil de Setembro/2026
**Racional (Aula 2 de Tecnologia):**
- **Retrato vs. Histórico (`assinaturas.csv` $\times$ `eventos_assinatura.csv`):** A tabela `assinaturas.csv` é uma foto em `30/09/2026`. Dos **2.250 assinantes ativos com `renovacao_automatica == 'nao'`** no retrato de `30/09`, reconstruindo pelos eventos até `31/07/2026`: apenas **1.427** já estavam com renovação desligada, **754** ainda estavam com ela **ligada** e **69** ainda nem assinavam (**823 casos = 37% de vazamento do futuro** se usarmos o retrato!).
- **Funil Contábil de Setembro/2026 (`pagamentos.csv`):**
  1. Soma bruta em `2026-09` sem filtrar status: `R$ 509.588`
  2. Filtrando `status == 'aprovado'` (fuso original): `R$ 499.503`
  3. Harmonizando fuso `UTC` (`Z`) das lojas para Horário de Brasília (`UTC-3`): `R$ 499.402`
  4. Expurgando cobranças aprovadas em duplicidade no mesmo ciclo/dia: **`R$ 492.572` (Faturamento Bruto)**
  5. Descontando taxas de intermediação (`15%` lojas, `3,5%` cartão, `0,99%` Pix): **`R$ 436.442` (Faturamento Líquido Oficial)**


In [9]:
import sys
sys.path.append('../src')
from limpeza_qualidade_dados import harmonizar_data_utc_para_br, limpar_valor_monetario, TAXAS_CANAL

# 7.1 Auditoria de Vazamento Temporal em 31/07/2026
df_ev = pd.read_csv(os.path.join(RAW_DIR, 'eventos_assinatura.csv'))
df_ev['data_hora_br'] = harmonizar_data_utc_para_br(df_ev['data_hora'])

ativos_renov_nao = ativas_pagantes[ativas_pagantes['renovacao_automatica'] == 'nao'].copy()
ativos_renov_nao['inicio_dt'] = pd.to_datetime(ativos_renov_nao['inicio'])
corte_jul = pd.Timestamp('2026-07-31 23:59:59')

ev_jul = df_ev[
    (df_ev['user_id'].isin(ativos_renov_nao['user_id']))
    & (df_ev['data_hora_br'] <= corte_jul)
    & (df_ev['evento'].isin(['assinatura_iniciada', 'renovacao', 'renovacao_automatica_desligada', 'renovacao_automatica_religada']))
].sort_values(['user_id', 'data_hora_br'])
ult_ev_jul = ev_jul.groupby('user_id')['evento'].last()

n_nao_ass_jul = int((ativos_renov_nao['inicio_dt'] > corte_jul).sum())
ja_ass_jul = ativos_renov_nao[ativos_renov_nao['inicio_dt'] <= corte_jul]
n_desligada_jul = int((ja_ass_jul['user_id'].map(ult_ev_jul) == 'renovacao_automatica_desligada').sum())
n_ligada_jul = len(ja_ass_jul) - n_desligada_jul

print(f"[Vazamento Temporal em 31/07/2026] Total 'nao' no retrato 30/09: {len(ativos_renov_nao):,} -> "
      f"Já desligada: {n_desligada_jul:,} | Ainda ligada: {n_ligada_jul:,} | Não assinava: {n_nao_ass_jul:,} "
      f"(Erro/Vazamento = {n_ligada_jul + n_nao_ass_jul:,} / {(n_ligada_jul + n_nao_ass_jul)/len(ativos_renov_nao)*100:.0f}%)")

# 7.2 Funil de Faturamento de Setembro/2026
df_pag['valor_num'] = limpar_valor_monetario(df_pag['valor'])
df_pag['data_hora_br'] = harmonizar_data_utc_para_br(df_pag['data_hora'])

m_set_str = df_pag['data_hora'].astype(str).str.startswith('2026-09')
v0 = df_pag.loc[m_set_str, 'valor_num'].sum()
v1 = df_pag.loc[m_set_str & (df_pag['status'] == 'aprovado'), 'valor_num'].sum()

aprov_br = df_pag[(df_pag['status'] == 'aprovado') & (df_pag['data_hora_br'] >= '2026-09-01') & (df_pag['data_hora_br'] <= '2026-09-30 23:59:59')].copy()
v2 = aprov_br['valor_num'].sum()

aprov_br['dia_br'] = aprov_br['data_hora_br'].dt.date
aprov_sem_dup = aprov_br.sort_values(['user_id', 'data_hora_br', 'tentativa']).drop_duplicates(subset=['user_id', 'valor_num', 'dia_br'], keep='first').copy()
v3 = aprov_sem_dup['valor_num'].sum()
v4 = (aprov_sem_dup['valor_num'] * (1.0 - aprov_sem_dup['metodo'].map(TAXAS_CANAL).fillna(0.0))).sum()

df_funil_fat = pd.DataFrame([
    {'Etapa do Funil Contábil (Setembro/2026)': '0. Soma bruta em 2026-09 (todos os status)', 'Valor Calculado': f'R$ {v0:,.0f}', 'Gabarito Aula 2': 'R$ 509.588'},
    {'Etapa do Funil Contábil (Setembro/2026)': '1. Apenas status == aprovado (fuso original)', 'Valor Calculado': f'R$ {v1:,.0f}', 'Gabarito Aula 2': 'R$ 499.503'},
    {'Etapa do Funil Contábil (Setembro/2026)': '2. Harmonizado para Horário de Brasília (UTC-3)', 'Valor Calculado': f'R$ {v2:,.0f}', 'Gabarito Aula 2': 'R$ 499.402'},
    {'Etapa do Funil Contábil (Setembro/2026)': '3. Expurgadas duplicatas no mesmo dia (Bruto)', 'Valor Calculado': f'R$ {v3:,.0f}', 'Gabarito Aula 2': 'R$ 492.572'},
    {'Etapa do Funil Contábil (Setembro/2026)': '4. Descontadas taxas de intermediação (Líquido)', 'Valor Calculado': f'R$ {v4:,.0f}', 'Gabarito Aula 2': 'R$ 436.442'},
])
display(df_funil_fat)


[Vazamento Temporal em 31/07/2026] Total 'nao' no retrato 30/09: 2,250 -> Já desligada: 1,427 | Ainda ligada: 754 | Não assinava: 69 (Erro/Vazamento = 823 / 37%)


,Etapa do Funil Contábil (Setembro/2026),Valor Calculado,Gabarito Aula 2
0,0. Soma bruta em 2026-09 (todos os status),"R$ 509,588",R$ 509.588
1,1. Apenas status == aprovado (fuso original),"R$ 499,503",R$ 499.503
2,2. Harmonizado para Horário de Brasília (UTC-3),"R$ 499,402",R$ 499.402
3,3. Expurgadas duplicatas no mesmo dia (Bruto),"R$ 492,572",R$ 492.572
4,4. Descontadas taxas de intermediação (Líquido),"R$ 436,442",R$ 436.442
